In [ ]:
import pandas as pd

from dotenv import load_dotenv
import os
load_dotenv()

In [2]:
data = pd.read_csv("data/2025-02-27_data_production.csv")

In [3]:
new_data = data[['Batch_name','Date','Technicien_id','Batch_KC8','Masse_KC8','Batch_THF_ref','Volume_THF',
      'BaG_T','Vitesse_agitation',
      'Heure_debut','Heure_ajout2','Lab_HR','Lab_T','QC_Conc_OGD','QC_Categorie']]

In [4]:
data_CQ = data[['Batch_name','QC_Conc_OGD','QC_Categorie']]
new_data.columns = ['Batch_OGD_name','Batch_OGD_date','Batch_OGD_Technicien','Batch_OGD_KC8_batch',
                'Batch_OGD_KC8_masse','Batch_OGD_THF_batch','Batch_OGD_THF_Volume','Batch_OGD_Temperature',
                'Batch_OGD_Agitation','Batch_OGD_heure_debut','Batch_OGD_heure_fin',
                'Batch_OGD_room_HR','Batch_OGD_room_T','Batch_OGD_Stock','Batch_OGD_Analyses']
new_data[['Batch_OGD_Stock','Batch_OGD_Analyses']] = 0

/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_3901/2406454623.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data[['Batch_OGD_Stock','Batch_OGD_Analyses']] = 0


In [ ]:
import mysql.connector as bdd_connect

BDD_CW = bdd_connect.connect(host='localhost',
                            user=os.getenv('LOCAL_DB_USER'),
                            password=os.getenv('LOCAL_DB_PASS'),
                            database=os.getenv('CW_DB_NAME'),
                            port=3306)
curseur = BDD_CW.cursor()

# Query avec docstring pour eviter SQL insertion
to_execute = f"SELECT Technicien_id, Initiales_tech FROM Techniciens"
curseur.execute(to_execute)

# Fetch data 
rows=curseur.fetchall()
curseur.close()
BDD_CW.close()

        # Export
df = dict(pd.DataFrame(rows, columns=[i[0] for i in curseur.description]).values)


In [6]:
new_data.loc[:,'Batch_OGD_Technicien'] = new_data['Batch_OGD_Technicien'].apply(lambda x : df[x])
new_data.loc[:,'Batch_OGD_Analyses'] = new_data['Batch_OGD_Analyses'].astype(str)
new_data.loc[:,'Batch_OGD_THF_batch'] = 'to_fill'

/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_3901/1810534463.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['IT' 'IT' 'IT' 'CD' 'IT' 'IT' 'CD' 'IT' 'CD' 'IT' 'IT' 'CD' 'IT' 'RS'
 'CD' 'IT' 'IT' 'CD' 'IT' 'IT' 'CD' 'IT' 'IT' 'CD' 'IT' 'IT' 'IT' 'CD'
 'IT' 'MM' 'IT' 'IT' 'FB' 'IT' 'FB' 'IT' 'IT' 'MM' 'CD' 'IT' 'CD' 'JP'
 'CD' 'JP' 'IT' 'JP' 'JP' 'IT' 'JP' 'JP' 'CD' 'JP' 'JP' 'CD' 'JP' 'JP'
 'JP' 'JP' 'CD' 'JP' 'JP' 'CD' 'JP' 'CD' 'JP' 'CD' 'JP' 'JP' 'CD' 'JP'
 'JP' 'JP' 'JP' 'CD' 'JP' 'CD' 'CD' 'JP' 'CD' 'CD' 'JP' 'CD' 'CD' 'JP'
 'JP' 'CD' 'JP' 'CD' 'JP' 'JP' 'CD' 'JP' 'JP' 'CD' 'JP' 'CD' 'JP' 'JP'
 'JP' 'JP' 'JP' 'JP' 'JP' 'JP' 'JP' 'CD' 'CD' 'JP' 'JP' 'NM' 'JP' 'NM'
 'CD' 'JP' 'NM' 'NM' 'JP' 'NM' 'NM' 'JP' 'CD' 'NM' 'NM' 'CD' 'JP' 'MM'
 'CD' 'CD' 'NM' 'NM' 'NM' 'CD' 'NM' 'CD' 'NM' 'NM' 'NM' 'CD' 'NM' 'CD'
 'NM' 'NM' 'JP' 'NM' 'CD' 'JP' 'JP' 'NM' 'CD' 'JP' 'NM' 'NM' 'JP' 'JP'
 'JP' 'CD' 'JP' 'JP'

In [7]:
import json
import requests

headers = {
'accept': 'application/json',
'Content-Type': 'application/json'}

for i in range(new_data.shape[0]):
    to_bdd = new_data.iloc[i,:].fillna(0).to_dict()

    response = requests.post('http://127.0.0.1:8000/OGD/', headers=headers, data=json.dumps(to_bdd))


In [9]:
response.content

b'{"Batch_OGD_id":280,"Batch_OGD_name":"2430D","Batch_OGD_date":"2024-07-22T00:00:00","Batch_OGD_Technicien":"JP","Batch_OGD_KC8_batch":"K28","Batch_OGD_KC8_masse":20.0645,"Batch_OGD_THF_batch":"to_fill","Batch_OGD_THF_Volume":500,"Batch_OGD_Temperature":30.0,"Batch_OGD_Agitation":230,"Batch_OGD_heure_debut":"2024-07-22T08:15:00","Batch_OGD_heure_fin":"2024-07-29T09:10:00","Batch_OGD_room_HR":54.0,"Batch_OGD_room_T":21.0,"Batch_OGD_Stock":0.0,"Batch_OGD_Analyses":"0"}'

In [8]:
for i in new_data:
    print(i)

Batch_OGD_name
Batch_OGD_date
Batch_OGD_Technicien
Batch_OGD_KC8_batch
Batch_OGD_KC8_masse
Batch_OGD_THF_batch
Batch_OGD_THF_Volume
Batch_OGD_Temperature
Batch_OGD_Agitation
Batch_OGD_heure_debut
Batch_OGD_heure_fin
Batch_OGD_room_HR
Batch_OGD_room_T
Batch_OGD_Stock
Batch_OGD_Analyses
